In [ ]:
import os
import pandas as pd
import neurokit2 as nk
import numpy as np

# Define input and output directories
training_input_dir = r"D:\Amrita\Sem-4\Machine Learning Lab\End Sem Project\Excel\1. Original\Train"
testing_input_dir = r"D:\Amrita\Sem-4\Machine Learning Lab\End Sem Project\Excel\1. Original\Test"
training_output_dir = r"D:\Amrita\Sem-4\Machine Learning Lab\End Sem Project\Excel\2. HRV\Train"
testing_output_dir = r"D:\Amrita\Sem-4\Machine Learning Lab\End Sem Project\Excel\2. HRV\Test"

# Ensure output directories exist
os.makedirs(training_output_dir, exist_ok=True)
os.makedirs(testing_output_dir, exist_ok=True)

SAMPLING_RATE = 200  # Hz

def extract_features(input_dir, output_dir):
    for file in os.listdir(input_dir):
        if file.endswith(".csv"):
            file_path = os.path.join(input_dir, file)
            df = pd.read_csv(file_path)
            
            # Separate ECG data and labels
            ecg_data = df.iloc[:, :-1]  # All columns except last (ECG signals)
            labels = df.iloc[:, -1]  # Last column (labels)
            
            all_features = []
            
            for i, row in ecg_data.iterrows():
                try:
                    ecg_cleaned = nk.ecg_clean(row.values, sampling_rate=SAMPLING_RATE)
                    peaks, _ = nk.ecg_peaks(ecg_cleaned, sampling_rate=SAMPLING_RATE)
                    hrv_features = nk.hrv(peaks, sampling_rate=SAMPLING_RATE).dropna(axis=1)
                    
                    # Compute statistical features
                    stats_features = {
                        "mean": np.mean(row),
                        "std": np.std(row),
                        "variance": np.var(row),
                        "skewness": pd.Series(row).skew(),
                        "kurtosis": pd.Series(row).kurtosis()
                    }
                    
                    stats_df = pd.DataFrame([stats_features])
                    combined_features = pd.concat([stats_df, hrv_features], axis=1)
                    
                    # Add label
                    combined_features["label"] = labels.iloc[i]
                    all_features.append(combined_features)
                except Exception as e:
                    print(f"Feature extraction error for {file} row {i}: {e}")
                    continue
            
            if all_features:
                feature_df = pd.concat(all_features, ignore_index=True)
                output_file = os.path.join(output_dir, f"{os.path.splitext(file)[0]}_features.csv")
                feature_df.to_csv(output_file, index=False)
                print(f"✅ Features saved: {output_file}")

# Extract features for training and testing datasets
extract_features(training_input_dir, training_output_dir)
extract_features(testing_input_dir, testing_output_dir)

print("🚀 Feature extraction complete!")

C:\Users\Jayanth C R\AppData\Local\Programs\Python\Python39\lib\site-packages\neurokit2\hrv\hrv_nonlinear.py:529: NeuroKitWarning: DFA_alpha2 related indices will not be calculated. The maximum duration of the windows provided for the long-term correlation is smaller than the minimum duration of windows. Refer to the `scale` argument in `nk.fractal_dfa()` for more information.
  warn(
C:\Users\Jayanth C R\AppData\Local\Programs\Python\Python39\lib\site-packages\neurokit2\complexity\entropy_multiscale.py:349: RuntimeWarning: invalid value encountered in scalar divide
  mse = np.trapz(mse) / len(mse)
C:\Users\Jayanth C R\AppData\Local\Programs\Python\Python39\lib\site-packages\neurokit2\complexity\entropy_multiscale.py:349: RuntimeWarning: invalid value encountered in scalar divide
  mse = np.trapz(mse) / len(mse)
C:\Users\Jayanth C R\AppData\Local\Programs\Python\Python39\lib\site-packages\neurokit2\complexity\entropy_multiscale.py:349: RuntimeWarning: invalid value encountered in scala